# S6E9 · 从 0.94570 继续：复用 CatBoost，补跑 LightGBM

**已知基线**：你上传的 `03_0.94570.ipynb` 保存了 CatBoost 五折 OOF AUC **0.9456182325**；截图中的 Public Score 为 **0.94570**。这两个数来自旧结果，不是这个 notebook 的新成绩。

本轮只做一件事：保留旧 CatBoost 概率，按原 V3 的 LightGBM 特征/参数补跑五折，再比较旧 CatBoost、LightGBM、75/25 与 50/50 概率融合。**不重新训练 CatBoost，不覆盖旧目录，不自动向 Kaggle 提交。**

本 notebook 已做代码检查和小型合成数据功能测试；**没有在你的真实竞赛数据上完成新训练，也没有新增排行榜分数**。

依赖：`numpy pandas scikit-learn lightgbm`。不自动安装。运行前保留原 `data` 和 `outputs/v3_20260925_180457_278867`；只把 notebook 放进项目里即可。旧 `.ipynb` 无需被新文件导入。

**操作顺序**：先运行第 1–3 节检查路径与旧结果；确认通过后运行第 4 节训练；最后运行第 5 节比较。直接 Run All 会包含第 4 节训练。

原 notebook 的文字“未执行训练 / Public 0.94555”是生成时留下的旧说明；本次审查以其中实际保存的训练日志和你的 0.94570 截图为准。

## 1. 加载原有训练与编码函数

下面保留你 V3 的算法：原来的行内数字特征、LightGBM 内层五折目标编码、相同外层五折和参数；仅把新训练目录前缀改为 `v4_lightgbm_`。函数定义不会开始训练。

目标编码仍严格在外层训练折内部计算；验证/测试只使用外层训练折的映射。不要为了省时间在整个训练集上先算一份目标编码。

原训练器保存模型、每折预测与编码来源的训练行索引；**模型文件本身不包含全部特征/目标编码过程**，以后推理必须配合相同代码与训练数据重建映射。

In [6]:
"""S6E9: fixed-fold CatBoost; optional LightGBM with cross-fitted target encoding.

Default: python s6e9_train_v3.py
Optional: python s6e9_train_v3.py --models catboost lightgbm
Dependencies: numpy pandas scikit-learn catboost; optional lightgbm.
No training occurs when this module is imported.
"""
import argparse
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

TARGET = "Will_Buy_EV"
NUMERIC = ["Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",
           "Charging_Stations_Near_Home", "Charging_Stations_Near_Work",
           "Environmental_Concern_Level"]


def features(frame, model_name):
    """Row-local features only: no labels or full-data target statistics."""
    x = frame.drop(columns=["id", TARGET], errors="ignore").copy()
    for col, scale, positions in [("Age", 1, 2), ("Annual_Income_USD", 1, 6),
                                 ("Daily_Commute_km", 10, 3),
                                 ("Charging_Stations_Near_Home", 1, 2),
                                 ("Charging_Stations_Near_Work", 1, 2)]:
        values = np.rint(frame[col].to_numpy() * scale).astype(np.int64)
        for digit in range(positions):
            x[f"{col}_digit_{digit}"] = (values // 10**digit % 10).astype(np.int8)
    income = np.rint(frame["Annual_Income_USD"]).astype(np.int64)
    x["income_last_2"] = income % 100
    x["income_last_3"] = income % 1000
    if model_name == "catboost":
        for col in NUMERIC:
            scale = 10 if col == "Daily_Commute_km" else 1
            x[f"{col}_as_category"] = np.rint(frame[col] * scale).astype(np.int64).astype(str)
    else:
        x["income_prefix_100"] = income // 100
        x["income_prefix_1000"] = income // 1000
    return x


def encode_fold(train_x, train_y, valid_x, test_x, seed, smoothing=1.0):
    """Inner OOF for training TE; outer-training-only maps for validation/test."""
    a, b, c = [d.copy().reset_index(drop=True) for d in (train_x, valid_x, test_x)]
    labels = np.asarray(train_y)
    inner = list(StratifiedKFold(5, shuffle=True, random_state=seed).split(a, labels))
    cats = a.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    keys = NUMERIC + cats + ["income_prefix_100", "income_prefix_1000"]
    for col in keys:
        encoded = np.full(len(a), np.nan)
        for fit_idx, hold_idx in inner:
            subset = pd.DataFrame({"key": a[col].iloc[fit_idx].to_numpy(), "y": labels[fit_idx]})
            stats = subset.groupby("key")["y"].agg(["sum", "count"])
            prior = labels[fit_idx].mean()
            mapping = (stats["sum"] + smoothing * prior) / (stats["count"] + smoothing)
            encoded[hold_idx] = a[col].iloc[hold_idx].map(mapping).fillna(prior).to_numpy()
        stats = pd.DataFrame({"key": a[col], "y": labels}).groupby("key")["y"].agg(["sum", "count"])
        prior = labels.mean()
        mapping = (stats["sum"] + smoothing * prior) / (stats["count"] + smoothing)
        a[f"{col}_te"] = encoded
        b[f"{col}_te"] = b[col].map(mapping).fillna(prior).to_numpy()
        c[f"{col}_te"] = c[col].map(mapping).fillna(prior).to_numpy()
        assert np.isfinite(encoded).all()
    # Label-free frequency: exclude a training row's own count contribution.
    counts = a["Annual_Income_USD"].value_counts()
    a["income_frequency"] = (a["Annual_Income_USD"].map(counts) - 1) / (len(a) - 1)
    for frame in (b, c):
        frame["income_frequency"] = frame["Annual_Income_USD"].map(counts).fillna(0) / len(a)
    for col in cats:
        dtype = pd.CategoricalDtype(categories=pd.unique(a[col]))
        for frame in (a, b, c):
            frame[col] = frame[col].astype(dtype)
    return a, b, c


def run(project="/Users/yw/unsw/Kaggle", models=("catboost",), threads=-1):
    models = tuple(models)
    if not models or len(set(models)) != len(models) or not set(models) <= {"catboost", "lightgbm"}:
        raise ValueError("models must contain distinct choices from catboost/lightgbm")
    # Check requested dependencies before starting any expensive training.
    if "catboost" in models:
        from catboost import CatBoostClassifier
    if "lightgbm" in models:
        try:
            import lightgbm as lgb
        except ImportError as error:
            raise ImportError("Optional LightGBM is unavailable; install lightgbm or run CatBoost only.") from error

    project = Path(project)
    train = pd.read_csv(project / "data/train.csv")
    test = pd.read_csv(project / "data/test.csv")
    assert train.drop(columns=TARGET).columns.tolist() == test.columns.tolist()
    assert train.id.is_unique and test.id.is_unique and not train.id.isin(test.id).any()
    assert not train.isna().any().any() and not test.isna().any().any()
    assert set(train[TARGET].unique()) == {"No", "Yes"}
    for frame in (train, test):
        assert np.allclose(frame.Annual_Income_USD, frame.Annual_Income_USD.round())
        assert np.allclose(frame.Daily_Commute_km * 10, (frame.Daily_Commute_km * 10).round())
    y = train[TARGET].map({"No": 0, "Yes": 1}).astype("int8")
    folds = list(StratifiedKFold(5, shuffle=True, random_state=42).split(train, y))
    out = project / "outputs" / ("v4_lightgbm_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    out.mkdir(parents=True, exist_ok=False)
    (out / "config.json").write_text(json.dumps({
        "models": models, "folds": 5, "seed": 42, "threads": threads,
        "train_rows": len(train), "test_rows": len(test),
        "note": "LightGBM/encoding and blend are unvalidated candidates; compare their OOF results.",
    }, indent=2))
    print("Output directory:", out, flush=True)
    all_oof, all_test, records = {}, {}, []

    def save_submission(name, probabilities):
        assert len(probabilities) == len(test)
        assert np.isfinite(probabilities).all()
        assert ((probabilities >= 0) & (probabilities <= 1)).all()
        pd.DataFrame({"id": test.id.to_numpy(), TARGET: probabilities}).to_csv(
            out / f"submission_{name}.csv", index=False)

    for name in models:
        x, xt = features(train, name), features(test, name)
        assert x.columns.tolist() == xt.columns.tolist()
        cats = x.select_dtypes(include=["object", "string", "category"]).columns.tolist()
        oof = np.full(len(train), np.nan)
        coverage = np.zeros(len(train), dtype=np.int8)
        test_sum = np.zeros(len(test))
        for fold, (ti, vi) in enumerate(folds, start=1):
            print(f"\n========== {name} | Fold {fold}/5 ==========", flush=True)
            a, b, c = x.iloc[ti], x.iloc[vi], xt
            if name == "catboost":
                # Same features/parameters as the successful single-fold candidate.
                model = CatBoostClassifier(
                    iterations=2000, learning_rate=0.03, depth=7,
                    loss_function="Logloss", eval_metric="AUC", random_seed=41 + fold,
                    thread_count=threads, allow_writing_files=False, verbose=400,
                )
                model.fit(a, y.iloc[ti], cat_features=cats, eval_set=(b, y.iloc[vi]),
                          early_stopping_rounds=150, use_best_model=True)
                model.save_model(str(out / f"{name}_fold{fold}.cbm"))
                best = int(model.get_best_iteration()) + 1
            else:
                a, b, c = encode_fold(a, y.iloc[ti], b, c, seed=1041 + fold)
                bins = [1023 if col == "Annual_Income_USD" else 255 for col in a.columns]
                model = lgb.LGBMClassifier(
                    n_estimators=5000, learning_rate=0.05, max_depth=5, num_leaves=31,
                    min_child_samples=100, reg_lambda=5.0, max_bin_by_feature=bins,
                    objective="binary", metric="auc", random_state=41 + fold, n_jobs=threads,
                    verbosity=-1, deterministic=True, force_col_wise=True,
                )
                model.fit(a, y.iloc[ti], eval_set=[(b, y.iloc[vi])], eval_metric="auc",
                          callbacks=[lgb.early_stopping(200, first_metric_only=True), lgb.log_evaluation(400)])
                model.booster_.save_model(str(out / f"{name}_fold{fold}.txt"))
                best = int(model.best_iteration_)
                # Persist fold-specific TE maps' source indices, for reproducible later inference.
                np.save(out / f"{name}_fold{fold}_train_indices.npy", ti)
            assert np.array_equal(model.classes_, [0, 1])
            vp = model.predict_proba(b)[:, 1]
            tp = model.predict_proba(c)[:, 1]
            assert np.isfinite(vp).all() and np.isfinite(tp).all()
            oof[vi], coverage[vi] = vp, coverage[vi] + 1
            test_sum += tp
            auc = float(roc_auc_score(y.iloc[vi], vp))
            records.append({"model": name, "fold": fold, "auc": auc, "best_trees": best})
            np.savez_compressed(out / f"{name}_fold{fold}_predictions.npz",
                                valid_indices=vi, valid_ids=train.id.iloc[vi].to_numpy(),
                                valid_prediction=vp, test_ids=test.id.to_numpy(), test_prediction=tp)
            pd.DataFrame(records).to_csv(out / "fold_scores.csv", index=False)
            print("Fold AUC:", auc, flush=True)
        assert (coverage == 1).all() and np.isfinite(oof).all()
        all_oof[name], all_test[name] = oof, test_sum / len(folds)
        pd.DataFrame({"id": train.id.to_numpy(), "target": y.to_numpy(), "prediction": oof}).to_csv(
            out / f"oof_{name}.csv", index=False)
        save_submission(name, all_test[name])
        print(name, "OOF AUC:", roc_auc_score(y, oof), flush=True)

    if len(models) == 2:
        # One predeclared blend, avoiding an extensive weight search on the same OOF.
        all_oof["blend_50_50"] = (all_oof["catboost"] + all_oof["lightgbm"]) / 2
        all_test["blend_50_50"] = (all_test["catboost"] + all_test["lightgbm"]) / 2
        save_submission("blend_50_50", all_test["blend_50_50"])
        pd.DataFrame({"id": train.id.to_numpy(), "target": y.to_numpy(),
                      "prediction": all_oof["blend_50_50"]}).to_csv(out / "oof_blend_50_50.csv", index=False)
    scores = {name: float(roc_auc_score(y, prediction)) for name, prediction in all_oof.items()}
    best_name = max(scores, key=scores.get)
    save_submission("best_oof", all_test[best_name])
    (out / "summary.json").write_text(json.dumps({
        "oof_auc": scores, "selected": best_name,
        "selection_note": "Selected on development OOF; this is not an independent audit or a Kaggle score.",
    }, indent=2))
    print("\nOOF comparison:", scores)
    print("Selected:", best_name, "\nCandidate:", out / "submission_best_oof.csv")
    return out




## 2. 路径与旧结果核对工具

默认路径来自你的已保存日志。旧结果缺失、ID/标签不一致、折划分不同、CSV 不等于每折预测的汇总，都会先报错，不会悄悄混用结果。

这能核对旧运行的标签、预测和划分，但不能追溯证明旧原始特征从未被修改：V3 没有保存原 CSV 的哈希。请保留当时原始的 `train.csv` / `test.csv`；本版本会记录其哈希供后续复用核对。

In [7]:
import sys
import hashlib
import importlib.metadata


def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def input_hashes(project):
    root = Path(project) / "data"
    names = ["train.csv", "test.csv", "sample_submission.csv"]
    return {name: file_hash(root / name) for name in names if (root / name).is_file()}


def probabilities(values, expected_length, description):
    p = np.asarray(values, dtype=float)
    require(p.shape == (expected_length,), f"{description}: wrong prediction shape")
    require(np.isfinite(p).all(), f"{description}: NaN or infinity")
    require(((p >= 0) & (p <= 1)).all(), f"{description}: probabilities outside [0, 1]")
    return p


def align_csv(path, ids, columns):
    """Do not blend CSVs by their incidental row order: align explicitly by id."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"缺少文件：{path}。请先确认原 outputs 目录；尚未开始新训练。")
    frame = pd.read_csv(path)
    require(set(["id", *columns]).issubset(frame.columns), f"{path.name}: missing columns")
    require(frame.id.notna().all() and frame.id.is_unique, f"{path.name}: invalid/duplicate ids")
    wanted = pd.Index(np.asarray(ids))
    require(len(frame) == len(wanted) and wanted.isin(frame.id).all(), f"{path.name}: id sets differ")
    return frame.set_index("id").loc[wanted, columns].reset_index()


def load_project(project):
    project = Path(project)
    for name in ("train.csv", "test.csv"):
        if not (project / "data" / name).is_file():
            raise FileNotFoundError(f"找不到 {project / 'data' / name}；请检查 PROJECT。")
    train = pd.read_csv(project / "data/train.csv")
    test = pd.read_csv(project / "data/test.csv")
    require(TARGET in train, "train.csv is missing the target")
    require(train.drop(columns=TARGET).columns.tolist() == test.columns.tolist(), "Train/test schema differs")
    require(train.id.is_unique and test.id.is_unique, "Duplicate id")
    require(not train.id.isin(test.id).any(), "Train/test ids overlap")
    require(not train.isna().any().any() and not test.isna().any().any(), "Missing values found")
    require(set(train[TARGET].unique()) == {"No", "Yes"}, "Expected No/Yes labels")
    y = train[TARGET].map({"No": 0, "Yes": 1}).astype("int8")
    require(y.value_counts().min() >= 10, "Too few rows per class for nested five-fold validation")
    folds = list(StratifiedKFold(5, shuffle=True, random_state=42).split(train, y))
    sample = project / "data/sample_submission.csv"
    if sample.exists():
        sample_frame = pd.read_csv(sample)
        require(sample_frame.columns.tolist() == ["id", TARGET], "Unexpected submission columns")
        require(sample_frame.id.is_unique and len(sample_frame) == len(test), "Invalid sample submission ids")
        require(pd.Index(test.id).isin(sample_frame.id).all(), "Sample submission has different ids")
    return train, test, y, folds


def read_run(run_dir, model_name, train, test, y, folds, expected_auc=None):
    """Verify saved OOF, test averages, and the actual five validation splits."""
    run_dir = Path(run_dir)
    config_path = run_dir / "config.json"
    if not config_path.is_file():
        raise FileNotFoundError(f"找不到 {config_path}；请把 BASELINE_DIR 指向原训练结果目录。")
    config = json.loads(config_path.read_text())
    require(config.get("folds") == 5 and config.get("seed") == 42, "Run does not use V3 folds/seed")
    require(model_name in config.get("models", []), f"This run does not contain {model_name}")
    require(config.get("train_rows") == len(train) and config.get("test_rows") == len(test), "Dataset sizes differ")
    oof_frame = align_csv(run_dir / f"oof_{model_name}.csv", train.id, ["target", "prediction"])
    test_frame = align_csv(run_dir / f"submission_{model_name}.csv", test.id, [TARGET])
    require(np.array_equal(oof_frame.target.to_numpy(), y.to_numpy()), "OOF labels do not match current train.csv")
    oof = probabilities(oof_frame.prediction, len(train), "OOF")
    test_pred = probabilities(test_frame[TARGET], len(test), "Test")
    reconstructed = np.full(len(train), np.nan)
    coverage = np.zeros(len(train), dtype=np.int8)
    test_sum = np.zeros(len(test))
    fold_records = []
    for fold, (_, vi) in enumerate(folds, start=1):
        path = run_dir / f"{model_name}_fold{fold}_predictions.npz"
        if not path.is_file():
            raise FileNotFoundError(f"缺少 {path}；无法核对原五折，不会静默混用其他划分。")
        with np.load(path, allow_pickle=False) as saved:
            require(np.array_equal(saved["valid_indices"], vi), f"Fold {fold}: split differs")
            require(np.array_equal(saved["valid_ids"], train.id.iloc[vi]), f"Fold {fold}: train order/ids differ")
            require(np.array_equal(saved["test_ids"], test.id), f"Fold {fold}: test order/ids differ")
            vp = probabilities(saved["valid_prediction"], len(vi), f"Fold {fold} OOF")
            tp = probabilities(saved["test_prediction"], len(test), f"Fold {fold} test")
            reconstructed[vi] = vp
            coverage[vi] += 1
            test_sum += tp
            fold_records.append({"fold": fold, "auc": float(roc_auc_score(y.iloc[vi], vp))})
    require((coverage == 1).all(), "OOF coverage is not exactly once per row")
    require(np.allclose(reconstructed, oof, atol=1e-12, rtol=0), "OOF CSV differs from fold predictions")
    require(np.allclose(test_sum / 5, test_pred, atol=1e-12, rtol=0), "Test CSV is not this run's five-fold average")
    score = float(roc_auc_score(y, oof))
    if expected_auc is not None:
        require(abs(score - expected_auc) < 1e-8,
                f"旧 OOF AUC 是 {score:.10f}，不是本次预期的 {expected_auc:.10f}；请检查目录版本。")
    return {"oof": oof, "test": test_pred, "auc": score, "fold_scores": pd.DataFrame(fold_records)}


def compare_and_save(project, baseline_dir, lightgbm_dir, train, test, y, folds, old, new, hashes):
    """Four predeclared candidates only; this is development selection, not an audit."""
    require(input_hashes(project) == hashes, "Input CSV files changed during this session; stop and inspect")
    out = Path(project) / "outputs" / ("v4_compare_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    out.mkdir(parents=True, exist_ok=False)
    # Include the old baseline so a weaker LightGBM does not automatically replace it.
    candidates = [("catboost_baseline", 1.0), ("lightgbm", 0.0),
                  ("blend_cat75_lgb25", 0.75), ("blend_cat50_lgb50", 0.50)]
    records, per_fold, candidate_test = [], [], {}
    base_fold = [roc_auc_score(y.iloc[vi], old["oof"][vi]) for _, vi in folds]
    for name, weight in candidates:
        op = weight * old["oof"] + (1 - weight) * new["oof"]
        tp = probabilities(weight * old["test"] + (1 - weight) * new["test"], len(test), name)
        auc = float(roc_auc_score(y, op))
        deltas = []
        for fold, (_, vi) in enumerate(folds, start=1):
            fa = float(roc_auc_score(y.iloc[vi], op[vi]))
            delta = float(fa - base_fold[fold - 1])
            deltas.append(delta)
            per_fold.append({"candidate": name, "fold": fold, "auc": fa, "delta_vs_catboost": delta})
        records.append({"candidate": name, "catboost_weight": weight,
                        "oof_auc": auc, "delta_vs_catboost": auc - old["auc"],
                        "improved_folds": sum(d > 0 for d in deltas),
                        "worst_fold_delta": min(deltas)})
        pd.DataFrame({"id": train.id.to_numpy(), "target": y.to_numpy(), "prediction": op}).to_csv(
            out / f"oof_{name}.csv", index=False)
        submission = pd.DataFrame({"id": test.id.to_numpy(), TARGET: tp})
        sample_path = Path(project) / "data/sample_submission.csv"
        if sample_path.exists():
            sample_ids = pd.read_csv(sample_path, usecols=["id"])
            submission = sample_ids.merge(submission, on="id", how="left", validate="one_to_one", sort=False)
        probabilities(submission[TARGET], len(test), "Submission")
        submission.to_csv(out / f"submission_{name}.csv", index=False)
        candidate_test[name] = submission
    # max keeps the first candidate on an exact tie (the old baseline).
    best = max(records, key=lambda row: row["oof_auc"])
    comparison = pd.DataFrame(records).sort_values("oof_auc", ascending=False, kind="stable").reset_index(drop=True)
    comparison.to_csv(out / "comparison.csv", index=False)
    pd.DataFrame(per_fold).to_csv(out / "comparison_by_fold.csv", index=False)
    candidate_test[best["candidate"]].to_csv(out / "submission_next_candidate.csv", index=False)
    (out / "continuation_summary.json").write_text(json.dumps({
        "baseline_dir": str(baseline_dir), "lightgbm_dir": str(lightgbm_dir),
        "baseline_oof_auc": old["auc"], "user_reported_baseline_public_auc": 0.94570,
        "selected_by_development_oof": best, "candidates": records,
        "input_sha256": hashes,
        "note": "OOF was used for early stopping and model/weight selection; not an independent test. No new Kaggle score is known.",
    }, indent=2), encoding="utf-8")
    print(comparison.to_string(index=False, float_format=lambda v: f"{v:.8f}"))
    print("\n比较目录：", out)
    print("OOF 选择的候选：", best["candidate"])
    print("候选 CSV：", out / "submission_next_candidate.csv")
    if best["candidate"] == "catboost_baseline":
        print("本轮候选均未超过旧 CatBoost。保留原 0.94570 提交，不必为了新版本而重新提交。")
    else:
        print("这是开发 OOF 最优候选，不保证 Public/Private 分数更高。请同时检查逐折变化，不做大规模权重搜索。")
    return out, comparison


In [8]:
PROJECT = Path("/Users/yw/unsw/Kaggle")
BASELINE_DIR = PROJECT / "outputs/v3_20260925_180457_278867"
EXPECTED_BASELINE_AUC = 0.9456182325266642
THREADS = -1  # 保持原设定。需要同时使用电脑时可改成 4。

# 第一次保留 None；已经完整跑过本 notebook 的 LightGBM 时，填入其输出目录，跳过重训。
# 这里只复用【完整】结果，不支持某一折训练到一半的自动恢复。
LGB_RESULT_DIR = None
# 示例（请用程序实际打印的目录，不要直接复制占位符）：
# LGB_RESULT_DIR = PROJECT / "outputs/v4_lightgbm_实际时间戳"


## 3. 先检查，不训练

正常情况下应打印旧 OOF AUC 约 `0.9456182325`。若报错缺少文件，请调整 `PROJECT` / `BASELINE_DIR` 或恢复旧输出，不要随便换成另一次训练目录。

`sample_submission.csv` 存在时会检查它并用其 ID 顺序导出；不存在时保留原 `test.csv` 的 ID 顺序。

In [13]:
# 这一格只检查，不训练。
try:
    import lightgbm as lgb
except (ImportError, OSError) as error:
    raise RuntimeError(
        "当前 Jupyter 内核无法导入 lightgbm。请先在这个内核安装/修复后重启内核；"
        "缺 Python 包时可在单独一格运行 %pip install lightgbm。"
    ) from error

train, test, y, folds = load_project(PROJECT)
old = read_run(BASELINE_DIR, "catboost", train, test, y, folds,
               expected_auc=EXPECTED_BASELINE_AUC)
DATA_HASHES = input_hashes(PROJECT)
print("当前 Python：", sys.executable)
print("旧结果检查通过；CatBoost OOF AUC：", old["auc"])
print(old["fold_scores"].to_string(index=False))
print("本轮不会重训 CatBoost。接下来运行第 4 节才会训练 LightGBM。")


当前 Python： /opt/miniconda3/envs/dl/bin/python
旧结果检查通过；CatBoost OOF AUC： 0.9456182325266642
 fold      auc
    1 0.944681
    2 0.945343
    3 0.946578
    4 0.945892
    5 0.945626
本轮不会重训 CatBoost。接下来运行第 4 节才会训练 LightGBM。


## 4. 只补跑 LightGBM 五折

参数保持原候选：5000 轮上限、学习率 0.05、深度 5、最多 31 个叶节点、连续 200 轮无提升早停。每折的最佳轮次由验证集决定。

第一次运行 `LGB_RESULT_DIR = None`。完整训练完成后程序会在同一内核记住目录；重启内核后，可以将第 2 节 `LGB_RESULT_DIR` 改成实际打印的目录，跳过训练。**中断在某一折时，原训练器没有自动断点续训；重新启动会新建目录并从五折的第一折重跑。**

不要因为这一节打印 `Selected: lightgbm` 就认为它比旧模型强：它在这一节是唯一参选模型，最终比较在下一节。

In [15]:
# 只有这里会启动训练；最多新训练一套 LightGBM 五折。
if LGB_RESULT_DIR is None:
    require(input_hashes(PROJECT) == DATA_HASHES, "检查后数据文件已改变")
    lightgbm_dir = run(project=PROJECT, models=("lightgbm",), threads=THREADS)
    require(input_hashes(PROJECT) == DATA_HASHES, "训练过程中数据文件已改变")
    (lightgbm_dir / "data_fingerprints.json").write_text(json.dumps(DATA_HASHES, indent=2))
    versions = {name: importlib.metadata.version(name)
                for name in ("numpy", "pandas", "scikit-learn", "lightgbm")}
    (lightgbm_dir / "package_versions.json").write_text(json.dumps(versions, indent=2))
    LGB_RESULT_DIR = lightgbm_dir  # 同一内核再次运行此格会复用完整结果。
else:
    lightgbm_dir = Path(LGB_RESULT_DIR)
    require(lightgbm_dir.resolve() != BASELINE_DIR.resolve(), "不能把旧 CatBoost 目录当成 LightGBM 目录")
    fingerprint_path = lightgbm_dir / "data_fingerprints.json"
    if not fingerprint_path.is_file():
        raise FileNotFoundError(f"缺少 {fingerprint_path}。请选择本 notebook 已完成的 LightGBM 目录。")
    require(json.loads(fingerprint_path.read_text()) == DATA_HASHES, "复用目录对应的数据文件版本不同")
    print("复用完整的 LightGBM 结果：", lightgbm_dir)

new = read_run(lightgbm_dir, "lightgbm", train, test, y, folds)
print("LightGBM OOF AUC：", new["auc"])
print("注意：此训练目录里的 submission_best_oof.csv 只在 LightGBM 本轮内选择。")
print("请继续运行第 5 节，才能与旧 CatBoost 一起比较。")


Output directory: /Users/yw/unsw/Kaggle/outputs/v4_lightgbm_20260930_163526_079778

========== lightgbm | Fold 1/5 ==========


/opt/miniconda3/envs/dl/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[400]	valid_0's auc: 0.944996
Early stopping, best iteration is:
[426]	valid_0's auc: 0.945
Evaluated only: auc
Fold AUC: 0.9450000800852427

========== lightgbm | Fold 2/5 ==========


/opt/miniconda3/envs/dl/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[400]	valid_0's auc: 0.945558
Early stopping, best iteration is:
[568]	valid_0's auc: 0.94559
Evaluated only: auc
Fold AUC: 0.9455901125635516

========== lightgbm | Fold 3/5 ==========


...

Training until validation scores don't improve for 200 rounds
[400]	valid_0's auc: 0.946798
Early stopping, best iteration is:
[495]	valid_0's auc: 0.946827
Evaluated only: auc
Fold AUC: 0.9468266726944491

========== lightgbm | Fold 4/5 ==========


/opt/miniconda3/envs/dl/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[400]	valid_0's auc: 0.946274
Early stopping, best iteration is:
[496]	valid_0's auc: 0.946303
Evaluated only: auc
Fold AUC: 0.9463030519746207

========== lightgbm | Fold 5/5 ==========


/opt/miniconda3/envs/dl/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[400]	valid_0's auc: 0.946015
[800]	valid_0's auc: 0.946008
Early stopping, best iteration is:
[608]	valid_0's auc: 0.946045
Evaluated only: auc
Fold AUC: 0.9460449515173676
lightgbm OOF AUC: 0.9459463810009869

OOF comparison: {'lightgbm': 0.9459463810009869}
Selected: lightgbm 
Candidate: /Users/yw/unsw/Kaggle/outputs/v4_lightgbm_20260930_163526_079778/submission_best_oof.csv
LightGBM OOF AUC： 0.9459463810009869
注意：此训练目录里的 submission_best_oof.csv 只在 LightGBM 本轮内选择。
请继续运行第 5 节，才能与旧 CatBoost 一起比较。


## 5. 用相同 OOF 比较四个候选，导出提交文件

只比较预先规定的四个方案，不做密集权重搜索。75/25 表示 `0.75 × CatBoost + 0.25 × LightGBM`；50/50 同理。

先按 ID 对齐预测，然后计算整体 OOF AUC，并记录逐折相对旧 CatBoost 的变化。`improved_folds` 是五个验证折中有几个提高；它**不是显著性检验**。OOF 同时用于早停和选择，因此这些结果是开发分数，不是独立评估。

新目录 `v4_compare_时间戳` 中：`comparison.csv` 是总体比较，`comparison_by_fold.csv` 是逐折变化，`submission_next_candidate.csv` 是开发 OOF 选出的候选。旧模型也参与选择；新方案没超过旧模型时保留旧模型，不需要重复提交同一批预测。

In [16]:
comparison_dir, comparison = compare_and_save(
    PROJECT, BASELINE_DIR, lightgbm_dir, train, test, y, folds, old, new, DATA_HASHES
)
comparison


        candidate  catboost_weight    oof_auc  delta_vs_catboost  improved_folds  worst_fold_delta
blend_cat50_lgb50       0.50000000 0.94598624         0.00036801               5        0.00032103
         lightgbm       0.00000000 0.94594638         0.00032815               5        0.00024666
blend_cat75_lgb25       0.75000000 0.94585286         0.00023463               5        0.00020918
catboost_baseline       1.00000000 0.94561823         0.00000000               0        0.00000000

比较目录： /Users/yw/unsw/Kaggle/outputs/v4_compare_20260930_163707_027699
OOF 选择的候选： blend_cat50_lgb50
候选 CSV： /Users/yw/unsw/Kaggle/outputs/v4_compare_20260930_163707_027699/submission_next_candidate.csv
这是开发 OOF 最优候选，不保证 Public/Private 分数更高。请同时检查逐折变化，不做大规模权重搜索。


,candidate,catboost_weight,oof_auc,delta_vs_catboost,improved_folds,worst_fold_delta
0,blend_cat50_lgb50,0.50,0.945986,0.000368,5,0.000321
1,lightgbm,0.00,0.945946,0.000328,5,0.000247
2,blend_cat75_lgb25,0.75,0.945853,0.000235,5,0.000209
3,catboost_baseline,1.00,0.945618,0.000000,0,0.000000


## 怎么读结果、怎么决定下一步

以旧 OOF **0.9456182325** 为比较基线，而不是把它与 Public **0.94570** 跨数据集直接比输赢。即使新 OOF 更高，也不保证新的 Public 或 Private 更高。很小的提升尤其要看逐折变化；不要为了追逐 Public 的偶然变化反复调权重。

若 LightGBM 或融合没有提高，本轮仍然得到了有效的负结果：保持原方案。下一项可单独测试把 CatBoost 迭代上限从 2000 提到 3500，并保持特征、学习率和相同五折；这需要新训练，**本 notebook 没有启动该实验**。

参考：
- [竞赛评分与截止时间](https://www.kaggle.com/competitions/playground-series-s6e9/overview)
- [scikit-learn：ROC AUC](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)
- [scikit-learn：目标编码的内部交叉拟合](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_target_encoder_cross_val.html)
- [LightGBM：early_stopping](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.early_stopping.html)

算法主体延续用户上传的 V3 实现；新增部分负责旧运行核对、复用、ID 对齐、数据版本记录及四候选比较。